# Multi-iterations


# Install & Imports

In [ ]:
!pip install -q ipywidgets openai
from google.colab import userdata
import base64
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, Markdown, HTML, clear_output
from openai import OpenAI
import re

# SETUP & CONFIGURATION

In [ ]:
try:
    openai_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=openai_key)
    print("✅ OpenAI Client (GPT-4o) Initialized")
except Exception as e:
    client = None
    print(f"⚠️ Warning: Could not fetch 'OPENAI_API_KEY'. Error: {e}")

# Architect Prompt

In [ ]:
ARCHITECT_PROMPT = """
You are a Lead Solution Architect following the Attribute-Driven Design (ADD) v3.0 process.
You must strictly follow these steps for every iteration:
- Step 2: Establish iteration goal by selecting drivers.
- Step 3: Choose elements of the system to refine.
- Step 4: Choose design concepts (Patterns/Tactics).
- Step 5: Instantiate elements, define interfaces, and sketch views.
- Step 6: Record design decisions in tables.
- Step 7: Analyze the design to ensure drivers are satisfied.

CORE GUIDELINES:
- Focus on high-performance (CQRS), reliability (Event Bus), and Cloud-native patterns.
- Output the Architecture Document content in Markdown format.
- Include Design Decision Tables and Rationale.
- Use Mermaid.js (inside ```mermaid blocks) to generate architectural diagrams directly in the document.
- PAUSE after every ADD step for human review.
"""

class ArchState:
    def __init__(self):
        self.reset()

    def reset(self):
        self.current_iteration = 1
        self.current_step = 2
        self.arch_doc = "# Architecture Document\n*Waiting for setup...*"
        self.system_context = ""

state = ArchState()



#  DIAGRAM RENDERER

In [ ]:
def call_architect_logic(task_prompt):
    if not client: return "❌ OpenAI Client not initialized."
    messages = [
        {"role": "system", "content": ARCHITECT_PROMPT},
        {"role": "user", "content": f"DOCUMENT:\n{state.arch_doc}\n\nCONTEXT:\n{state.system_context}\n\nTASK:\n{task_prompt}"}
    ]
    try:
        response = client.chat.completions.create(model="gpt-4o", messages=messages, temperature=0.1)
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Architect Error: {str(e)}"

def render_diagrams_from_text(text):
    # Slightly relaxed regex to catch variations in AI whitespace
    mermaid_blocks = re.findall(r"```mermaid\s*(.*?)\s*```", text, re.DOTALL)
    for i, code in enumerate(mermaid_blocks):
        chart_id = f"mermaid-{i}-{datetime.now().microsecond}"
        html_code = f"""
            <div style="border:1px solid #ddd; padding:15px; border-radius:8px; margin-top:10px; background-color:white; box-shadow: 2px 2px 5px #eee;">
                <strong style="color:#333;">📊 Architecture View</strong>
                <div class="mermaid" id="{chart_id}">{code}</div>
            </div>
            <script>
                if (typeof mermaid !== 'undefined') {{
                    mermaid.init(undefined, document.getElementById('{chart_id}'));
                }} else {{
                    let script = document.createElement('script');
                    script.src = "https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js";
                    script.onload = () => {{
                        mermaid.initialize({{startOnLoad: true, theme: 'neutral'}});
                        mermaid.init(undefined, document.getElementById('{chart_id}'));
                    }};
                    document.head.appendChild(script);
                }}
            </script>
        """
        display(HTML(html_code))

def create_download_link():
    b64 = base64.b64encode(state.arch_doc.encode()).decode()
    return f'<a href="data:file/markdown;base64,{b64}" download="Architecture_Document.md" style="color:white; background-color:#2196F3; padding:10px 20px; border-radius:5px; text-decoration:none; font-weight:bold;">📩 Download Final Document (.md)</a>'


# UI WIDGETS

In [ ]:
drivers_input = widgets.Textarea(
    value="[Enter your detailed context and quality attribute scenarios here...]",
    description='Context:', layout=widgets.Layout(width='98%', height='150px')
)
feedback_input = widgets.Text(placeholder='Optional: Provide feedback to the architect...', description='Feedback:', layout=widgets.Layout(width='98%'))
output = widgets.Output()
download_output = widgets.Output()

def run_setup(_):
    state.system_context = drivers_input.value
    with output:
        clear_output()
        print("🔄 Step 1: Architecting Iteration Plan (Dynamic Analysis)...")
        prompt = "Review the System Context and create a customized 6-iteration ADD Plan Table based on risk and priority. Then generate the Initial C4 Context description."
        text_result = call_architect_logic(prompt)
        state.arch_doc = text_result

        display(Markdown(text_result))
        render_diagrams_from_text(text_result)

    with download_output:
        clear_output()
        display(HTML(create_download_link()))

def run_next_step(_):
    with output:
        clear_output(wait=True)
        feedback = f"\nReviewer Feedback: {feedback_input.value}" if feedback_input.value else ""

        prompt = f"""
        Execute ADD Iteration {state.current_iteration}, Step {state.current_step}.
        Refer to your established Iteration Plan Table and execute the goals for this phase.
        {feedback}
        Update the Architecture Document accordingly.
        """

        print(f"🔄 Processing Iteration {state.current_iteration} | Step {state.current_step}...")
        text_result = call_architect_logic(prompt)

        # APPENDS TO THE DOC INSTEAD OF OVERWRITING
        state.arch_doc += f"\n\n---\n\n# Iteration {state.current_iteration} | Step {state.current_step}\n\n{text_result}"

        feedback_input.value = ""

        display(Markdown(f"# Iteration {state.current_iteration} | Step {state.current_step}"))
        display(Markdown(text_result))
        render_diagrams_from_text(text_result)

        if state.current_step < 7:
            state.current_step += 1
        else:
            state.current_step = 2
            state.current_iteration += 1

    with download_output:
        clear_output()
        display(HTML(create_download_link()))


# Application Display

In [ ]:

btn_setup = widgets.Button(description="1. Initialize Architecture", button_style='primary')
btn_next = widgets.Button(description="2. Execute Next ADD Step", button_style='success')
btn_setup.on_click(run_setup)
btn_next.on_click(run_next_step)

display(widgets.VBox([drivers_input, feedback_input, widgets.HBox([btn_setup, btn_next]), download_output, output]))

✅ OpenAI Client (GPT-4o) Initialized
